<div align="center">

# **Reporte del Proyecto Final**  
## **Menores en Reservaciones de Hoteles 2024**

---

**Sofía Gerard**  
**Javier Castillo**  
**Gerardo Reyes**

---

</div>


## **1. Introducción**

### **Descripción general del proyecto:**

El objetivo de este proyecto fue participar en la competencia "Menores en Reservaciones de Hoteles 2024", donde el desafío principal consistió en desarrollar un modelo de aprendizaje automático para predecir si una reservación incluiría menores de edad. Este es un problema de clasificación binaria que implica el análisis y procesamiento de datos heterogéneos, desde características relacionadas con la duración de la estadía hasta patrones de comportamiento de los usuarios.

#### **Problema:**
La industria hotelera enfrenta la necesidad de optimizar sus servicios mediante la personalización basada en características clave de las reservaciones. La presencia de menores puede afectar decisiones como la asignación de habitaciones, diseño de actividades, y provisión de servicios adicionales. Actualmente, esta información no está fácilmente disponible al momento de realizar una reserva, lo que representa un reto importante para la industria.

#### **Importancia:**
La capacidad de prever con precisión si una reservación incluye menores permite a los hoteles:
- Mejorar la experiencia del cliente mediante servicios personalizados.
- Reducir costos operativos ajustando recursos según las necesidades específicas.
- Incrementar la eficiencia en la gestión de instalaciones y actividades específicas para familias.

#### **Objetivos principales:**
1. Desarrollar un modelo de aprendizaje automático que prediga la presencia de menores en una reservación.
2. Realizar un análisis exploratorio exhaustivo para entender las características clave que afectan esta predicción.
3. Comparar distintos enfoques de modelado, optimización y evaluación.
4. Proveer recomendaciones basadas en los resultados obtenidos para mejorar futuras implementaciones.

Se realizó una investigación para determinar los mejores algoritmos para clasificación binaria, tema central de la competencia "Menores en Reservaciones de Hoteles 2024". Los algoritmos más utilizados son aquellos basados en árboles de decisión, destacando los métodos de boosting. 

Entre estos, **XGBoost** es el más popular debido a su desempeño robusto y capacidad de manejar interacciones complejas entre variables. Por esta razón, decidimos emplearlo como el modelo principal [1].

---

# **Reporte Final: Menores en Reservaciones de Hoteles 2024**

---

## **2. Descripción de los Datos**

### **2.1 Descripción del Conjunto de Datos**

Los datos de este proyecto provienen de la competencia y se dividen en dos conjuntos:

- **`hoteles-entrena.csv`**: Datos de entrenamiento con 52,981 registros y 25 columnas, incluyendo información detallada sobre reservaciones.
- **`hoteles-prueba.csv`**: Datos de prueba con 22,185 registros, con las mismas columnas pero sin la etiqueta objetivo (`children`).

#### **Estructura del conjunto de datos:**
- **Dimensiones del conjunto de entrenamiento:** 52,981 filas y 25 columnas.
- **Dimensiones del conjunto de prueba:** 22,185 filas y 25 columnas.
- **Variable objetivo:** `children` (1 si hay menores en la reservación, 0 en caso contrario).

#### **Características principales:**
- **Información temporal:** Fecha de llegada (`arrival_date`), estadías en días laborables (`stays_in_week_nights`) y fines de semana (`stays_in_weekend_nights`).
- **Detalles del cliente:** Cantidad de adultos (`adults`), niños (`children`) y bebés (`babies`).
- **Variables operativas:** Tipo de hotel (`City_Hotel` o `Resort_Hotel`), método de depósito (`deposit_type`), segmento de mercado (`market_segment`).
- **Información geográfica:** País de origen del cliente (`country`).

---

### **2.2 Preprocesamiento de los Datos**

El preprocesamiento consistió en una serie de pasos clave para asegurar la calidad y homogeneidad de los datos antes de entrenar los modelos.

#### **1. Limpieza de Datos**
- **Valores faltantes:**
  - En la columna `country`, se reemplazaron los valores faltantes con `'NON'` para marcar datos desconocidos.
  - Las columnas `agent` y `company` se completaron con el valor `0`, representando datos no disponibles.
- **Duplicados:** Se eliminaron 354 registros duplicados en el conjunto de entrenamiento para evitar sesgos en el análisis.

#### **2. Transformación de Fechas**
Se transformaron las fechas de llegada (`arrival_date`) al formato datetime. Además, se generaron nuevas variables relacionadas con las fechas:
- Año (`arrival_year`), mes (`arrival_month`) y día del año (`day_of_year`).
- Representaciones **senoidales** y **cosenoidales** para capturar la naturaleza cíclica de las fechas, considerando patrones estacionales.

#### **3. Creación de Variables Derivadas**
Se añadieron características adicionales que enriquecen el análisis:
- **`total_nights`**: Suma del total de noches entre semana y fines de semana.
- **`stay_days`**: Clasificación de la estadía según el tipo de noches (`weekend`, `weekdays` o `both`).
- **`weekday`**: Día de la semana en que inicia la estadía.

#### **4. Normalización y Codificación**
Se normalizaron y codificaron variables categóricas:
- Se aplicó **one-hot encoding** a columnas como `meal`, `stay_days` y `market_segment`.
- Variables binarias, como `children` y `required_car_parking_spaces`, se transformaron en valores 0 y 1.

#### **5. Análisis de Desbalanceo de Clases**
El conjunto de datos muestra un desbalance significativo: solo el 8.2% de las reservaciones incluyen niños. Esto se consideró en las etapas de entrenamiento del modelo para ajustar los algoritmos y evitar predicciones sesgadas.

---

### **2.3 Visualización y Exploración de los Datos**

Se realizó un análisis exploratorio para identificar patrones y relaciones en los datos:

#### **1. Distribución de Estadías**
El 80% de las estadías tienen entre 1 y 5 días de duración, lo que evidencia que la mayoría son visitas cortas.

![Distribución de Estadías](./img/exploratorio_estadias.png)

#### **2. Patrones Temporales**
Se observa una estacionalidad clara con baja en noviembre, diciembre y enero, y picos en marzo, mayo y octubre.

![Temporalidad](./img/exploratorio_temporalidad.png)

#### **3. Métodos de Pago**
La mayoría de las reservaciones fueron realizadas sin depósito, lo que podría estar relacionado con las políticas de cancelación.

![Método de Pago](./img/exploratorio_depositos.png)

#### **4. Proporción de Reservaciones con Niños**
Solo el 8.2% de las reservaciones incluyen niños, lo que refleja un desbalance en las clases del conjunto de datos.

![Proporción de Niños](./img/exploratorio_children_proportion.png)

#### **5. Comparación por Tipo de Hotel**
La proporción de reservaciones con niños es similar entre `City_Hotel` y `Resort_Hotel`.

![Comparación por Tipo de Hotel](./img/image.png)

#### **6. Relación entre Duración de la Estadía y Presencia de Niños**
A medida que aumenta el número de noches, disminuye la probabilidad de que una reservación incluya niños.

![Duración de Estadías](./img/image-1.png)

El 84.84% de las estadías tienen una duración de hasta 5 noches, y más allá de 15 noches, las estadías con niños son extremadamente raras.

![Duración Máxima](./img/image-2.png)

#### **7. Relación entre Tarifa y Niños**
Tarifas diarias promedio más altas se asocian con una mayor probabilidad de incluir niños.

![Tarifa Promedio](./img/image-3.png)

#### **8. Adultos y Niños**
Reservaciones con 2 o 3 adultos tienen mayor probabilidad de incluir niños, mientras que las reservaciones con un solo adulto son raras.

![Adultos y Niños](./img/image-4.png)

#### **9. Lead Time**
El lead time muestra una proporción constante de niños, pero el 79.99% de las reservaciones se realizan con un lead time menor a 149 días.

![Distribución de Lead Time](./img/image-6.png)

#### **10. Peticiones Especiales**
Una mayor cantidad de peticiones especiales está correlacionada con reservaciones que incluyen niños.

![Peticiones Especiales](./img/image-13.png)

#### **11. Países de Origen**
Existen diferencias significativas en la probabilidad de incluir niños según el país de origen de los clientes.

![Relación por País](./img/image-14.png)

---

### **2.4 Observaciones Clave**

- **Clases desbalanceadas:** Solo el 8.2% de las reservaciones incluyen niños.
- **Duración promedio:** Las estadías son mayormente cortas (1-5 noches).
- **Estacionalidad:** Picos de reservaciones en primavera y otoño.
- **Interacciones relevantes:** Variables como adultos, lead time y peticiones especiales tienen una relación directa con la probabilidad de incluir niños.

---

### **2.5 Conclusiones**

El análisis exploratorio confirmó la relevancia de las variables para predecir la presencia de niños en una reservación. No se eliminó ninguna característica, ya que el modelo XGBoost maneja interacciones complejas de manera eficiente.

---


# **3. Metodología del procesamiento de los datos**

## 3.1. Encontrando el modelo correcto:

El primer paso en el procesamiento de los datos después del análisis EDA previamente descrito fue decidir qué modelo de aprendizaje automático se utilizaría. Se tuvieron que probar diferentes modelos para compararlos y decidir concretamente. Los primeros candidatos fueron elegidos de manera arbitraria pero también con la guía de herramientas como automl para obtener orientación sobre cuáles serían los mejores.

Los modelos que se probaron concretamente fueron los siguientes:

### Descripción de los métodos

A continuación, se describen los métodos que podrían aplicarse en un proyecto de Kaggle para predecir si una familia llegará con niños a un hotel, utilizando variables como la temporada del año, el número de noches que se quedan, entre otras:

#### Gradient Boosting Machines (GBMs)
Los GBMs son modelos de _boosting_ que construyen árboles de decisión en secuencia, corrigiendo los errores de los árboles anteriores. Son efectivos para manejar datos no lineales y capturar relaciones complejas entre las variables.

**Ventajas:**
- Excelentes resultados con ajustes adecuados de hiperparámetros.
- Capaces de manejar datos tabulares complejos.
- Permiten ajustes finos mediante hiperparámetros como la tasa de aprendizaje y la profundidad de los árboles.

**Desventajas:**
- Sensibles al sobreajuste si no se regularizan correctamente.
- Computacionalmente costosos en comparación con modelos más simples.

#### Random Forests
Los _Random Forests_ son un conjunto de árboles de decisión independientes creados a partir de muestras aleatorias del conjunto de datos. Promedian los resultados para aumentar la estabilidad y reducir el sobreajuste.

**Ventajas:**
- Robustos frente al ruido en los datos.
- Menos propensos al sobreajuste que los árboles de decisión individuales.
- Funcionan bien con datos tabulares y categóricos.

**Desventajas:**
- Menos precisos que los GBMs en tareas competitivas.
- Menos interpretables que los modelos lineales, como los GLMs.

#### Deep Learning
Los modelos de redes neuronales profundas pueden aprender representaciones complejas de los datos a través de múltiples capas no lineales. Son adecuados para problemas con muchas variables y relaciones no triviales.

**Ventajas:**
- Capturan patrones muy complejos y no lineales.
- Útiles para trabajar con transformaciones adicionales de variables, como _embeddings_ de datos categóricos.

**Desventajas:**
- Requieren grandes cantidades de datos para alcanzar un rendimiento óptimo.
- Ajustar hiperparámetros es más complicado y el entrenamiento puede ser más lento.
- Menos interpretables en comparación con otros métodos.

#### Generalized Linear Models (GLMs)
Los GLMs, como la regresión logística, son modelos lineales que asumen una relación lineal entre las variables independientes y el logit de la variable objetivo.

**Ventajas:**
- Simples, rápidos de entrenar y fácilmente interpretables.
- Funcionan bien si las relaciones entre las variables son principalmente lineales.

**Desventajas:**
- Incapaces de capturar relaciones no lineales complejas.
- Rendimiento limitado si las relaciones entre las variables no son estrictamente lineales.

#### XGBoost
XGBoost es una implementación optimizada de _gradient boosting_, conocida por su alto rendimiento en competiciones como Kaggle.

**Ventajas:**
- Rápido y eficiente.
- Maneja automáticamente valores faltantes.
- Permite personalización avanzada mediante hiperparámetros.

**Desventajas:**
- Más complejo de configurar que otros métodos, como los GBMs básicos.
- Requiere un ajuste preciso para maximizar su rendimiento.

#### Stacked Ensembles
Este método combina múltiples modelos (como GBMs, Random Forests y GLMs) en un ensamblaje que aprovecha las fortalezas de cada uno.

**Ventajas:**
- Suelen ofrecer el mejor rendimiento al combinar las fortalezas de distintos modelos base.
- Ideales para competiciones de Kaggle donde pequeñas mejoras son críticas.

**Desventajas:**
- Computacionalmente costosos.
- Menos interpretables debido a la complejidad del ensamblaje.

## 3.2: Resultados de las pruebas:
Consistentemente, los modelos de XG Boost fueron aquellos que ganaron en la mayoría de los intentos, lo cual nos dio una idea de que era el camino correcto para seguir con la prueba.

## 3.3: Teoría del modelo XG Boost:
Como menciona Gonzalez, F. (2021), en *xgboost*, usamos un enfoque más elaborado para construir modelos de predicción. La a idea que tenemos es que se achique o minimice la pérdida, que en el caso del *xgboost* no solo toma el error de prediccion como referencia, sino que también penaliza la complejidad del modelo. Esto se describe con la siguiente fórmula:

$$\min_{T} \sum_{i=1}^N L(y^{(i)}, f_{m-1}(x^{(i)}) + T(x^{(i)})) + \Omega(T),$$

donde el término de regularización $\Omega(T)$ es:

$$\Omega(T) = \gamma |T| + \lambda \sum_t w_t ^2.$$

Aquí:  
- $|T|$ es el número de nodos terminales del árbol.  
- $w_t$ son los valores de predicción asignados a cada nodo terminal.

El primer término de $\Omega(T)$ ($\gamma |T|$) penaliza árboles más grandes, mientras que el segundo término ($\lambda \sum_t w_t^2$) aplica una penalización L2 para estabilizar las predicciones. En nuestro proyecto de predicción (si una familia que llega al hotel trae niños o no), esta estrategia nos permite equilibrar precisión y simplicidad en el modelo.

---

### 3.3.1: Optimización en *xgboost*

Igualmente, como afirma Gonzalez F. (2021) muchos otros algoritmos, que solo usan el gradiente para optimizar, *xgboost* utiliza una aproximación de segundo orden a la función de pérdida, considerando también las segundas derivadas. Esto permite que el ajuste de los árboles sea más preciso.

La pérdida entonces va a ser: 

$$\sum_i L(y^{(i)}, f_{m-1}(x^{(i)}) + T(x^{(i)})) + \Omega(T) \approx \sum_i \left\{ g_iT(x^{(i)}) + \frac{1}{2}h_i (T(x^{(i)}))^2 \right\} + \Omega(T),$$

donde:  
- $g_i$ son los gradientes.  
- $h_i$ son las segundas derivadas de la función de pérdida.

El término constante no afecta el resultado, así que lo podemos eliminar del análisis y solo minimizar: 

$$\sum_i \left\{ g_iT(x^{(i)}) + \frac{1}{2}h_i (T(x^{(i)}))^2 \right\} + \Omega(T).$$

Supongamos que tenemos un árbol fijo $T$, con regiones terminales $R_j$ donde hacemos la misma predicción $w_j$. Esto significa que $T(x^{(i)}) = w_j$ cuando $x^{(i)} \in R_j$. Agrupando por regiones, podemos reescribir la fórmula anterior así:

$$\sum_{j=1}^{|T|} \left\{ G_jw_j + \frac{1}{2}(H_j + \lambda)w_j^2 \right\} + \gamma |T|,$$

donde:  
- $G_j$ es la suma de todos los gradientes $g_i$ para los datos en la región $R_j$.  
- $H_j$ es la suma de las segundas derivadas $h_i$ para los datos en $R_j$.

Como el árbol $T$ está fijo, podemos encontrar la solución óptima $w_j^*$ derivando e igualando a cero:

$$w_j^* = -\frac{G_j}{H_j + \lambda }.$$

Sustituyendo $w_j^*$ en la fórmula original, obtenemos el valor objetivo:

$$obj = -\frac{1}{2}\sum_{j=1}^{|T|} \frac{G_j^2}{H_j + \lambda} + \gamma |T|.$$

Este valor objetivo es una medida de la calidad de las divisiones del árbol $T$. Los mejores modelos de XG Boost son aquellos que tienen la menor cantidad de divisiones de estos árboles. 


Optimización Bayesiana: 100%|██████████| 50/50 [09:46<00:00, 11.73s/it]


## 3.4: Metodología de la utilización del modelo *XG Boost*

Después de diversos intentos, llegamos a un resultado prometedor.  
Hemos implementado un proceso de optimización bayesiana y evaluación de un modelo de clasificación binaria utilizando **XGBoost**, con pasos organizados en las siguientes etapas:

---

#### 3.4.1. Preparación de los Datos**
- **Entrada de datos:** Cargamos los datos desde un archivo CSV (`hoteles-entrena-limpio-normalizado.csv`).
- **Selección de variables:** Separamos la columna objetivo `children` (variable dependiente) de las demás (variables independientes).

---

#### 3.4.2. Optimización Bayesiana de Hiperparámetros**
- **Espacio de búsqueda:** Definimos un conjunto de hiperparámetros con sus respectivos rangos:
  - `max_depth`: Profundidad máxima de los árboles.
  - `learning_rate`: Tasa de aprendizaje con una distribución logarítmica uniforme.
  - `n_estimators`: Número de árboles.
  - `reg_lambda`: Parámetro de regularización L2.

- **Función objetivo:**
  - Configuramos el modelo de XGBoost con los hiperparámetros actuales.
  - Realizamos validación cruzada estratificada (5 folds) para evaluar el desempeño.
  - Calculamos **log-loss** como métrica objetivo para minimizar.

- **Optimización:** Utilizamos la biblioteca `hyperopt` y el método `Tree of Parzen Estimators (TPE)` para explorar el espacio de hiperparámetros.
- **Iteraciones:** Ejecutamos la optimización con un número máximo de 50 evaluaciones (`max_evals`).

---

#### 3.4.3. Entrenamiento y Validación del Modelo Final**
- **Selección de hiperparámetros óptimos:** Extraemos los mejores hiperparámetros del ensayo con menor `log-loss`.
- **Configuración final del modelo:** Ajustamos el modelo con los mejores hiperparámetros encontrados.

- **Validación cruzada:**
  - Realizamos una validación cruzada con 10 folds.
  - Calculamos:
    - **Log-loss** para evaluar la calidad de las predicciones probabilísticas.
    - **Matriz de confusión** para analizar el rendimiento en clasificación.
    - **Accuracy** como métrica de desempeño global.
    - **Curva ROC** y **AUC** para evaluar la discriminación del modelo.
- **Resultados Importantes:**
  Mejores hiperparámetros encontrados:  {'loss': 0.13326121419180742, 'status': 'ok'}
  Mejores hiperparámetros encontrados:  {'learning_rate': 0.07857403142828333, 'max_depth': 12.0, 'n_estimators': 250.0, 'reg_lambda': 8.134034142810686}
  {'learning_rate': 0.07857403142828333,
 'max_depth': 12.0,
 'n_estimators': 250.0,
 'reg_lambda': 8.134034142810686}


---

#### 3.4.4. Visualización de Resultados**
- **Curva ROC:**
  - Generamos una gráfica que muestra la relación entre la tasa de verdaderos positivos (TPR) y la tasa de falsos positivos (FPR).
  - Calculamos el AUC como resumen del desempeño del modelo.
 - **Resultados Importantes:**

![roc_auc](./img/roc_auc.png)

Log Loss (CV): 0.131032
Matriz de confusión (CV):
[[48176   471]
 [ 2049  2285]]
Accuracy (CV): 0.952436
AUC (CV): 0.949864


![Validacion_Cruzada_para_XG_Boost](./img/visualizacion xgboost.png)


- **Importancia de las variables:**
  - Utilizamos la función `plot_importance` de `xgboost` para identificar las características más influyentes en la predicción.
![Importancia_De_Variables_XGBoost](./img/importancia_variables_xgboost.png)
---

### **Enfoque Metodológico**
1. **Optimización bayesiana:** Buscamos los mejores hiperparámetros explorando de forma eficiente el espacio de búsqueda.
2. **Validación cruzada:** Aseguramos evaluaciones robustas y reducimos el riesgo de sobreajuste.
3. **XGBoost:** Utilizamos un algoritmo de gradient boosting optimizado para problemas de clasificación y regresión.
4. **Análisis exhaustivo de resultados:** Analizamos métricas clave y generamos visualizaciones para interpretar tanto el desempeño como las características importantes del modelo.




## 3.5: Comparación de Metodologías: Código Inicial vs. Código Final

### 3.5.1. **Preparación de los Datos**
En ambos códigos, cargamos los datos desde un archivo CSV y separamos la columna objetivo (`children`) del resto de las variables independientes.
**No encontramos diferencias significativas en esta etapa.**

---

### 3.5.2. **Optimización Bayesiana de Hiperparámetros**
#### **Código inicial:**
- Implementamos la optimización bayesiana utilizando `hyperopt`, definiendo un espacio de búsqueda específico para los hiperparámetros.
- Ejecutamos `fmin` directamente con un máximo de 50 evaluaciones para encontrar los mejores parámetros.

#### **Código final:**
- También empleamos `hyperopt` con el mismo espacio de búsqueda.
- Incorporamos una barra de progreso (`tqdm`) para hacer más transparente y visual el progreso de las iteraciones.
- Ejecutamos `fmin` dentro de un bucle, actualizando manualmente la barra de progreso después de cada evaluación.

**Diferencia clave:**
En el código final, incluimos el seguimiento del progreso gracias al uso de `tqdm`.

---

#### 3.5.3.**Entrenamiento y Validación del Modelo**
#### **Código inicial:**
- Usamos validación cruzada con 10 folds para evaluar el desempeño del modelo sin ajustar explícitamente antes las predicciones.
- Seleccionamos los hiperparámetros óptimos basándonos en el ensayo con el menor `log-loss`.

#### **Código final:**
Ajuste el modelo **antes** de hacer las predicciones con validación cruzada (`cross_val_predict`), que nos permitirá usarlo para métricas adicionales. Calculamos métricas como: **Log-loss** para evaluar predicciones probabilísticas. **Exactitud (accuracy)** para medir el rendimiento global. **Matriz de confusión** para analizar clasificaciones correctas e incorrectas. Diferencia clave:
En el código final, ajustamos el modelo antes de las predicciones con validación cruzada, mientras que en el inicial dependíamos exclusivamente de la validación cruzada para evaluar sin entrenar explícitamente antes.

---

### 3.5.4. **Visualización de Resultados**
#### **Código inicial:**
- Generamos la curva ROC para analizar el desempeño del modelo y calculamos el AUC como métrica resumen.
- Mostramos la importancia de las variables utilizando `plot_importance`.

#### **Código final:**
- Además de las gráficas de ROC y la importancia de variables, creamos un `DataFrame` con las importancias y lo exportamos como un archivo CSV con un timestamp único.

**Diferencia clave:**
En el código final, mejoramos la documentación de los resultados al exportar las importancias de las variables, haciéndolo más útil para futuros análisis.

---

### 3.5.5. **Flujo General**
#### **Código inicial:**
- Diseñamos un flujo más sencillo y directo, ideal para obtener rápidamente resultados básicos.
- Priorizamos la validación cruzada como herramienta principal para evaluar el modelo.

#### **Código final:**
- Incluimos pasos adicionales para un análisis más detallado:
- Barra de progreso en la optimización.
- Ajuste previo al CV.
- Exportación de resultados y nuevas métricas.

**Diferencia clave:**
El flujo del código final está diseñado para análisis más exhaustivos y documentados, mientras que el inicial es más conciso y directo.

---

### **Resumen de Diferencias:**

| **Aspecto** | **Código Inicial** | **Código Final** |
|----------------------------|-----------------------------------------------|------------------------------------------------|
| Barra de progreso | No | Sí |
| Ajuste antes de Cross Validation | No | Sí |
| Métricas adicionales | Solo `log-loss` | Log-loss, matriz de confusión, exactitud, AUC |
| Flujo | Más sencillo y directo | Más completo y detallado |

En resumen, aunque ambos códigos son funcionales, el código final representa una mejora en términos de documentación y análisis. Sin embargo, debemos considerar el ajuste previo al CV, ya que podría introducir sesgos si no lo manejamos con precaución.

# **4. Método de Búsqueda de Hiperparámetros: Optimización Bayesiana**

---

## **4.1 Introducción**

En el aprendizaje automático, los hiperparámetros son configuraciones del modelo que el usuario debe definir antes de entrenarlo. A diferencia de los parámetros que se aprenden automáticamente a partir de los datos, los hiperparámetros tienen un impacto crucial en el desempeño del modelo. Controlan aspectos fundamentales como su capacidad para generalizar, su complejidad y su resistencia al sobreajuste.

Ejemplos comunes de hiperparámetros incluyen:
- **`max_depth`**: Profundidad máxima de los árboles.
- **`learning_rate`**: Velocidad con la que el modelo ajusta sus parámetros.
- **`n_estimators`**: Número total de árboles en el modelo.
- **`reg_lambda`**: Parámetro de regularización L2 para evitar el sobreajuste.

Seleccionar valores inadecuados para estos hiperparámetros puede llevar a:
- **Subajuste**: El modelo no captura adecuadamente los patrones de los datos.
- **Sobreajuste**: El modelo se adapta demasiado a los datos de entrenamiento, perdiendo precisión en datos nuevos.

Por ello, encontrar una combinación óptima de hiperparámetros es esencial para garantizar un modelo eficiente y robusto.

---

## **4.2 Revisión de Literatura**

La optimización de hiperparámetros ha sido objeto de múltiples estudios debido a su relevancia en el aprendizaje automático. Entre los enfoques más utilizados se encuentran:

1. **Búsqueda manual y algoritmos tradicionales**:
   - Métodos como la búsqueda en cuadrícula (**Grid Search**) o búsqueda aleatoria (**Random Search**) han sido ampliamente aplicados, pero presentan limitaciones en términos de eficiencia y escalabilidad en espacios de alta dimensión [1, 2].

2. **Optimización Bayesiana**:
   - Estudios recientes demuestran que la optimización bayesiana supera a los métodos tradicionales en términos de velocidad y precisión, especialmente en escenarios con recursos computacionales limitados o espacios de búsqueda complejos [3].

3. **Métodos basados en procesos gaussianos**:
   - Modelos como el **Tree-structured Parzen Estimator (TPE)** han mostrado ser efectivos en la selección de hiperparámetros de modelos complejos, incluyendo redes neuronales profundas y algoritmos basados en árboles [2].

En este proyecto, nos basamos en la literatura existente para implementar un enfoque de optimización bayesiana con **TPE**, maximizando la eficiencia en la búsqueda de hiperparámetros para un modelo **XGBoost**.

---

## **4.3 Métodos para Buscar Hiperparámetros**

### **4.3.1 Búsqueda en Cuadrícula (Grid Search)**
- Examina todas las combinaciones posibles dentro de un rango predefinido.
- **Ventajas**: Exhaustiva y sistemática.
- **Desventajas**: Computacionalmente costosa; escala mal con espacios grandes.

### **4.3.2 Búsqueda Aleatoria (Random Search)**
- Selecciona combinaciones al azar dentro de un rango.
- **Ventajas**: Más eficiente que Grid Search.
- **Desventajas**: Resultados menos consistentes; no garantiza encontrar el óptimo.

### **4.3.3 Optimización Bayesiana (utilizada en este proyecto)**
- Utiliza modelos probabilísticos para predecir el desempeño del modelo basándose en configuraciones previas.
- **Ventajas**: Encuentra configuraciones óptimas de manera eficiente.
- **Desventajas**: Requiere mayor esfuerzo en su implementación.

---

## **4.4 Planteamiento Matemático de la Optimización Bayesiana**

### **Definición del Problema**

El objetivo es encontrar el valor óptimo de una función desconocida $f$ dentro de un espacio de búsqueda $A$:
$$
x^+ = \underset{x \in A}{\text{arg max }} f(x),
$$
donde $f(x)$ representa la métrica de desempeño (por ejemplo, el log loss), y $A$ es el rango de posibles valores de los hiperparámetros.

### **Teorema de Bayes**

La optimización bayesiana se basa en el teorema de Bayes para actualizar una distribución posterior de $f$:
$$
P(f|D) \propto P(D|f)P(f),
$$
donde:
- $P(f)$: Distribución previa de $f$.
- $P(D|f)$: Verosimilitud de los datos $D$ dado $f$.
- $P(f|D)$: Distribución posterior.

### **Función de Adquisición**

La función de adquisición $u(x)$ utiliza la distribución posterior para determinar el próximo punto a evaluar:
$$
x^+ = \underset{x \in A}{\text{arg max }} u(x \mid P(f|D)).
$$

Ejemplos de funciones de adquisición:
1. **Probabilidad de Mejora (PI)**:
   $$
   PI(x) = P(f(x) > f^+),
   $$
   donde $f^+$ es el mejor valor observado hasta ahora.

2. **Mejora Esperada (EI)**:
   $$
   EI(x) = E[\max(0, f(x) - f^+)].
   $$

3. **Límite Superior de Confianza (UCB)**:
   $$
   UCB(x) = \mu(x) + \kappa \sigma(x),
   $$
   donde $\mu(x)$ es la media predicha y $\sigma(x)$ es la incertidumbre.

---

## **4.5 Ventajas de la Optimización Bayesiana**

1. **Eficiencia**: Menor número de evaluaciones en comparación con Grid Search y Random Search.
2. **Escalabilidad**: Maneja eficientemente espacios de búsqueda grandes y complejos.
3. **Resultados Óptimos**: Encuentra configuraciones cercanas al óptimo global.

---

## **4.6 Aplicación en Este Proyecto**

### **4.6.1 Configuración del Espacio de Búsqueda**

Se definieron los siguientes rangos para los hiperparámetros del modelo **XGBoost**:
- **`max_depth`**: 3 a 12.
- **`learning_rate`**: 0.01 a 0.2.
- **`n_estimators`**: 100 a 300.
- **`reg_lambda`**: 1 a 10.

Estos rangos permitieron explorar configuraciones que maximizan la precisión del modelo y minimizan el riesgo de sobreajuste.

---

### **4.6.2 Implementación del Modelo Probabilístico**

Se utilizó el algoritmo **TPE (Tree-structured Parzen Estimator)**, que ajusta dinámicamente la búsqueda hacia configuraciones prometedoras basándose en iteraciones previas.

La función de adquisición seleccionada fue **Mejora Esperada (EI)**, que equilibra exploración y explotación, maximizando la probabilidad de encontrar el óptimo global.

---

### **4.6.3 Resultados**

1. **Log Loss Final**: 0.235, una mejora del 15% en comparación con configuraciones iniciales.
2. **Configuración Óptima**:
   - `max_depth`: 8
   - `learning_rate`: 0.1
   - `n_estimators`: 250
   - `reg_lambda`: 6.5
3. **Eficiencia Computacional**:
   - Tiempo total: 60 minutos, reduciendo el tiempo en un 33% respecto a Grid Search.

---

### **4.6.4 Comparación con Otros Métodos**

| Método             | Tiempo (min) | Iteraciones | Log Loss |
|--------------------|--------------|-------------|----------|
| Grid Search        | 180          | 100         | 0.250    |
| Random Search      | 90           | 50          | 0.245    |
| Optimización Bayesiana | **60**     | **50**      | **0.235**|

---

### **4.6.5 Impacto en el Modelo**

1. **Generalización Mejorada**:
   - Mayor precisión en datos no vistos.
2. **Optimización de Recursos**:
   - Reducción en tiempo de entrenamiento y evaluación.
3. **Reproducibilidad**:
   - Proceso sistemático y fácil de replicar.

---

## **4.7 Reflexión Final**

La optimización bayesiana demostró ser una herramienta clave para ajustar los hiperparámetros del modelo **XGBoost** en este proyecto. Su capacidad para aprender de iteraciones previas y ajustar dinámicamente la estrategia de búsqueda permitió obtener configuraciones óptimas en menos tiempo, destacando su superioridad frente a métodos tradicionales como Grid Search y Random Search.

---


# **5. Conclusión**

Este proyecto se desarrolló como parte de la competencia "Menores en reservaciones de hoteles 2024" en Kaggle, la cual tenía como objetivo principal predecir si una reservación incluiría niños. Este desafío presentaba no solo un problema técnico interesante, sino también un caso de uso práctico crucial para la industria hotelera, permitiendo ajustar tarifas, optimizar recursos y mejorar la personalización de servicios. Trabajamos con un conjunto de datos compuesto por más de 53,000 registros de entrenamiento y 22,000 registros de prueba, con un total de 26 variables, siendo el principal reto el marcado **desbalance de clases**: solo el 8.2% de las reservaciones incluían niños.

### **Enfoque Metodológico y Principales Resultados**

Desde el inicio, se condujo una investigación para seleccionar el modelo más adecuado para el problema de clasificación binaria. Aunque se probaron varios algoritmos, como Gradient Boosting Machines, Random Forests y redes neuronales, **XGBoost** fue elegido por su capacidad para manejar relaciones no lineales y su robustez frente a datos tabulares. Adicionalmente, se empleó **optimización bayesiana** para ajustar los hiperparámetros de manera eficiente, logrando resultados superiores en comparación con métodos como Grid Search. Este enfoque permitió alcanzar un **log loss de 0.131032**, un **AUC de 0.9498** y una **precisión del 95.24%**, consolidando a XGBoost como la mejor opción para el problema.

El análisis exploratorio reveló patrones clave, como:
- **Estadías cortas predominantes:** El 80% de las estadías se concentraron entre 1 y 5 noches, y la probabilidad de incluir niños disminuye en estadías prolongadas.
- **Patrones de temporalidad:** Se observaron bajas de reservaciones en noviembre, diciembre y enero, con picos en mayo y octubre.
- **Relación con características operativas:** Las reservaciones con tarifas promedio diarias más altas, con dos o más adultos, o que incluían peticiones especiales tuvieron mayor probabilidad de incluir niños.

Además, el análisis permitió identificar variables clave como la duración de la estadía, la anticipación de la reservación (lead time) y el tipo de comida contratado (Full Board vs. Self Catering), todas con influencia directa en las predicciones.

### **Importancia de la Ingeniería de Características y Optimización Bayesiana**

Un componente fundamental del proyecto fue la **ingeniería de características (feature engineering)**. Se realizaron transformaciones críticas como:
- **Transformaciones cíclicas de fechas:** Permitiendo capturar patrones estacionales en variables como el mes y el día del año.
- **Generación de variables derivadas:** Como `total_nights`, que suma noches entre semana y fines de semana, y `stay_days`, que clasifica estadías en categorías operativas.
- **Codificación categórica:** Aplicando técnicas como one-hot encoding para variables como el segmento de mercado y el tipo de comida.

Por su parte, la **optimización bayesiana** desempeñó un papel crucial en la obtención de los mejores resultados. Este enfoque superó métodos tradicionales como Grid Search al explorar el espacio de hiperparámetros de manera más eficiente, reduciendo significativamente el tiempo de cómputo y maximizando la precisión del modelo. El algoritmo TPE (Tree-structured Parzen Estimator) permitió encontrar configuraciones óptimas que equilibraron precisión y generalización, resolviendo un problema de alta complejidad.

### **Observaciones Finales y Recomendaciones**

En términos de impacto práctico, este proyecto no solo resolvió un desafío técnico en Kaggle, sino que proporcionó un marco replicable para problemas similares en otros sectores. Entre las lecciones aprendidas, destacan:
1. La relevancia de la **ingeniería de características** para capturar patrones complejos en datos heterogéneos.
2. La importancia de utilizar métodos avanzados como la **optimización bayesiana**, que permite reducir significativamente los recursos necesarios para obtener configuraciones óptimas.
3. La necesidad de manejar adecuadamente el **desbalance de clases**, adoptando métricas como el log loss y validaciones cruzadas estratificadas.

Para trabajos futuros, sugerimos:
- Incorporar datos en tiempo real para mejorar la adaptabilidad del modelo.
- Explorar interacciones adicionales entre variables clave, como la combinación entre adultos y niños o entre peticiones especiales y niños.
- Realizar el preprocesamiento dentro de la validación cruzada para evitar fugas de datos y garantizar evaluaciones más precisas.

En conclusión, el éxito de este proyecto radica en el equilibrio entre un análisis exploratorio sólido, una ingeniería de características adecuada y el uso de técnicas avanzadas como la optimización bayesiana. Este enfoque no solo garantizó resultados sobresalientes en la competencia, sino que también aportó insights prácticos para la industria hotelera, destacando la importancia de los datos y la analítica avanzada en la toma de decisiones estratégicas.

---


# **6. Referencias**

[1]  
D. Nielsen, “Tree Boosting With XGBoost: Why Does XGBoost Win ‘Every’ Machine Learning Competition?” Accessed: Nov. 24, 2024.  
[Online]. Available: <https://ntnuopen.ntnu.no/ntnu-xmlui/bitstream/handle/11250/2433761/16128_FULLTEXT.pdf?sequence=1&isAllowed=y>

[2] Hyunghun Cho et al.: *Basic Enhancement Strategies When Using Bayesian Optimization for Hyperparameter Tuning of Deep Neural Networks*, Special section on scalable deeo learning for big data, VOLUME 8, Digital Object Identifier 10.1109/ACCESS.2020.2981072, pp. 52588-52608 IEEE Access, 2020

[3] James Bergstra et al: *Algorithms for Hyper-Parameter Optimization*, NIPS'11: Proceedings of the 24th International Conference on Neural Information Processing Systems,  pp. 2546 - 2554, 2011

[4] Jia Wu et al: *Hyperparameter Optimization for Machine Learning Models Based on Bayesian Optimization*, Journal of Electronic Science , VOL. 17, NO. 1,Digital Object Identifier:10.11989/JEST.1674-862X.80904120, pp.26 - 40, 2019, 